# 04 - QLoRA模型微调训练

在DGX Spark上使用QLoRA技术对基座模型进行TRIZ领域微调。

**硬件要求**: DGX Spark (128GB Unified Memory) | **训练时间**: ~15小时 (2 epochs)

**兼容性**: 本笔记本已适配 Transformers v5 + TRL v1.x + PEFT v0.18

## 4.1 环境检查与预飞行

验证训练环境是否满足要求。

In [ ]:
# 预飞行检查 — TRAIN-01
import sys
sys.path.append('/home/meerkat/mongoose_ai')

import torch
import os
from config import (
    BASE_MODEL, MODELS_DIR, DATA_DIR, OUTPUTS_DIR, RESULTS_DIR,
    CHECKPOINTS_DIR, QLORA_CONFIG, DATA_CONFIG
)

print("=" * 60)
print("预飞行检查")
print("=" * 60)

errors = []
warnings_list = []

# 1. 检查训练数据
print("\n[1/4] 检查训练数据...")
data_path = DATA_CONFIG['processed_data_dir']
if os.path.exists(data_path):
    print(f"  PASS 数据路径存在: {data_path}")
else:
    errors.append(f"训练数据不存在: {data_path}")
    print(f"  FAIL 训练数据: {data_path} 不存在")

# 2. 检查GPU内存
print("\n[2/4] 检查GPU内存...")
if torch.cuda.is_available():
    total_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    free_mem = total_mem - torch.cuda.memory_allocated() / (1024**3)
    print(f"  总内存: {total_mem:.1f} GB")
    print(f"  可用内存: {free_mem:.1f} GB")
    if free_mem < 60:
        warnings_list.append(f"可用GPU内存仅 {free_mem:.1f}GB")
        print(f"  WARN 可用内存不足: {free_mem:.1f}GB < 60GB")
    else:
        print(f"  PASS GPU内存充足: {free_mem:.1f}GB")
else:
    errors.append("CUDA不可用")
    print(f"  FAIL CUDA不可用")

# 3. 检查模型路径
print("\n[3/4] 检查模型路径...")
model_path = os.path.join(MODELS_DIR, BASE_MODEL.split('/')[-1])
if os.path.exists(model_path):
    print(f"  PASS 模型路径: {model_path}")
else:
    errors.append(f"模型路径不存在: {model_path}")
    print(f"  FAIL 模型路径: {model_path}")

# 4. 检查训练配置
print("\n[4/4] 检查训练配置...")
lora_dropout = QLORA_CONFIG['lora']['lora_dropout']
target_modules = QLORA_CONFIG['lora']['target_modules']

if lora_dropout == 0.0:
    print(f"  PASS lora_dropout=0.0")
else:
    warnings_list.append(f"lora_dropout={lora_dropout} (建议0.0)")
    print(f"  WARN lora_dropout={lora_dropout}")

if isinstance(target_modules, list) and len(target_modules) == 12:
    print(f"  PASS target_modules: {len(target_modules)}个模块")
else:
    warnings_list.append("target_modules数量异常")
    print(f"  WARN target_modules数量异常")

print(f"  save_steps={QLORA_CONFIG['training']['save_steps']}")
print(f"  num_train_epochs={QLORA_CONFIG['training']['num_train_epochs']}")
print(f"  learning_rate={QLORA_CONFIG['training']['learning_rate']}")

# 总结
print("\n" + "=" * 60)
if errors:
    print(f"预飞行检查失败: {len(errors)}个错误")
    for e in errors:
        print(f"  ERROR: {e}")
    print("=" * 60)
    raise RuntimeError("预飞行检查失败")
else:
    print("预飞行检查通过!")
    if warnings_list:
        print(f"警告: {len(warnings_list)}个")
        for w in warnings_list:
            print(f"  WARN: {w}")
    print("=" * 60)

## 4.2 加载配置和数据

加载处理后的数据集。

In [ ]:
# 加载处理后的数据集
from utils.data_utils import load_processed_dataset

data_path = DATA_CONFIG['processed_data_dir']
print(f"加载数据集: {data_path}")

dataset = load_processed_dataset(data_path)

print(f"\n数据集加载完成!")
for split_name, split_data in dataset.items():
    print(f"  {split_name}: {len(split_data)} 条样本")

# 验证数据集格式
sample = dataset['train'][0] if 'train' in dataset else None
if sample:
    print(f"\n样本字段: {list(sample.keys())}")
    print(f"instruction长度: {len(sample.get('instruction', ''))}")
    print(f"output长度: {len(sample.get('output', ''))}")

## 4.3 加载模型 (4-bit量化)

使用4-bit NF4量化加载模型。`training_utils.py` 已更新以自动检测DGX Spark统一内存。

In [ ]:
# 加载4-bit量化模型 — TRAIN-07
from utils.training_utils import load_model_and_tokenizer

model_path = os.path.join(MODELS_DIR, BASE_MODEL.split('/')[-1])

print(f"加载模型: {model_path}")
print("启用4-bit量化 (NF4) 以节省内存...")

model, tokenizer = load_model_and_tokenizer(
    model_name_or_path=model_path,
    quantization_config=QLORA_CONFIG['quantization'],
    device_map='auto',
    trust_remote_code=True,
)

print(f"\n模型加载完成!")
print(f"显存占用: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

## 4.4 配置QLoRA

使用显式12模块target_modules列表（非all-linear），lora_dropout=0.0。

**注意**: 如果 `prepare_qlora_model()` 导致OOM，在调用前运行 `model.config.use_cache = False`。

In [ ]:
# (可选) 验证target_modules配置
from utils.training_utils import find_all_linear_names, get_qwen36_target_modules

print("检测模型中的线性层模块...")
detected = find_all_linear_names(model)
print(f"\n检测到 {len(detected)} 个可用于LoRA的模块")

recommended = get_qwen36_target_modules()
missing = set(recommended) - set(detected)
extra = set(detected) - set(recommended)

if missing:
    print(f"警告: 推荐列表中有但模型中未检测到: {missing}")
if extra:
    print(f"提示: 模型中存在额外模块: {extra}")
if not missing and not extra:
    print("模块列表匹配完美!")

In [ ]:
# 创建QLoRA配置并准备模型
from utils.training_utils import setup_qlora_config, prepare_qlora_model

# 禁用KV缓存以防止prepare_qlora_model()时OOM (统一内存环境)
model.config.use_cache = False

lora_config = setup_qlora_config(
    r=QLORA_CONFIG['lora']['r'],
    lora_alpha=QLORA_CONFIG['lora']['lora_alpha'],
    target_modules=QLORA_CONFIG['lora']['target_modules'],
    lora_dropout=QLORA_CONFIG['lora']['lora_dropout'],
    use_rslora=QLORA_CONFIG['lora'].get('use_rslora', False),
)

model = prepare_qlora_model(model, lora_config)

print("\nQLoRA配置完成!")

## 4.5 配置训练参数

已适配 Transformers v5 API: `eval_strategy` 替代 `evaluation_strategy`，`group_by_length` 已移除，`load_best_model_at_end=False` 避免PEFT v0.18兼容性问题。

In [ ]:
# 训练参数 — 适配 Transformers v5 + PEFT v0.18
from utils.training_utils import setup_training_arguments

# Transformers v5 兼容配置
# - eval_strategy 替代 evaluation_strategy
# - group_by_length 已移除
# - load_best_model_at_end=False 避免PEFT weight converter bug
# - fp16=False 避免BF16 gradient scaler不兼容

training_args = setup_training_arguments(
    output_dir=QLORA_CONFIG['training']['output_dir'],
    num_train_epochs=QLORA_CONFIG['training']['num_train_epochs'],
    per_device_batch_size=QLORA_CONFIG['training']['per_device_train_batch_size'],
    gradient_accumulation_steps=QLORA_CONFIG['training']['gradient_accumulation_steps'],
    learning_rate=QLORA_CONFIG['training']['learning_rate'],
    warmup_ratio=QLORA_CONFIG['training']['warmup_ratio'],
    save_steps=QLORA_CONFIG['training']['save_steps'],
    eval_steps=QLORA_CONFIG['training']['eval_steps'],
    logging_steps=QLORA_CONFIG['training']['logging_steps'],
    save_total_limit=QLORA_CONFIG['training']['save_total_limit'],
    load_best_model_at_end=False,  # PEFT v0.18 + Transformers v5 不兼容
    metric_for_best_model=QLORA_CONFIG['training'].get('metric_for_best_model', 'eval_loss'),
    greater_is_better=QLORA_CONFIG['training'].get('greater_is_better', False),
    report_to=QLORA_CONFIG['training'].get('report_to', 'tensorboard'),
    fp16=False,  # BF16梯度在FP16 GradScaler中不兼容，禁用
    bf16=False,
)

print("训练参数配置完成!")
print(f"  输出目录: {training_args.output_dir}")
print(f"  训练轮数: {training_args.num_train_epochs}")
print(f"  有效batch_size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  学习率: {training_args.learning_rate}")
print(f"  优化器: {training_args.optim}")
print(f"  fp16: {training_args.fp16}")
print(f"  load_best_model_at_end: {training_args.load_best_model_at_end}")

## 4.6 创建Trainer并开始训练

使用SFTTrainer + formatting_func。已适配 TRL v1.x (`processing_class` 替代 `tokenizer`，移除 `max_seq_length` 和 `packing`)。

In [ ]:
# 创建Trainer并训练 — 适配 TRL v1.x
from utils.training_utils import CheckpointValidationCallback
from trl import SFTTrainer
import logging

logger = logging.getLogger(__name__)

system_message = DATA_CONFIG['chatml']['system_message']

def formatting_func(example):
    """将原始数据转换为对话文本"""
    instruction = example.get("instruction", "")
    input_text = example.get("input", "")
    output = example.get("output", "")
    sys_msg = example.get("system", system_message)
    full_question = f"{instruction}\n\n{input_text}" if input_text else instruction
    messages = [
        {"role": "system", "content": sys_msg},
        {"role": "user", "content": full_question},
        {"role": "assistant", "content": output},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

# 验证formatting_func
try:
    test_text = formatting_func(dataset['train'][0])
    print(f"格式化测试通过，样本长度: {len(test_text)} 字符")
except Exception as e:
    print(f"格式化测试失败: {e}")

# TRL v1.x: processing_class 替代 tokenizer; 移除 max_seq_length 和 packing
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    args=training_args,
    formatting_func=formatting_func,
)

# 添加checkpoint验证回调
checkpoint_callback = CheckpointValidationCallback(
    tokenizer=tokenizer,
    test_prompt="请解释TRIZ的分割原理及其应用场景。",
)
trainer.add_callback(checkpoint_callback)

print("Trainer创建完成 (含checkpoint验证回调)")
print("=" * 60)

# 开始训练
trainer.train()

print("=" * 60)
print("训练完成!")

# 打印验证结果摘要
if checkpoint_callback.validation_results:
    print(f"\nCheckpoint验证结果 ({len(checkpoint_callback.validation_results)}个):")
    passed = sum(1 for r in checkpoint_callback.validation_results if r.get('status') == 'PASSED')
    failed = sum(1 for r in checkpoint_callback.validation_results if r.get('status') == 'FAILED')
    print(f"  通过: {passed}, 失败: {failed}")
    for r in checkpoint_callback.validation_results:
        status = r.get('status', 'UNKNOWN')
        print(f"    step={r['step']}: {status}")

## 4.7 保存适配器

保存LoRA适配器（仅100-200MB）。元数据包括训练步数、最终loss、SHA-256哈希。

In [ ]:
# 保存LoRA适配器 — TRAIN-08
from utils.training_utils import save_adapter_only
import json

adapter_output_dir = os.path.join(MODELS_DIR, 'meerkat_triz_adapter_v1')

# 收集训练元数据
training_metadata = {
    'training_steps': trainer.state.global_step,
    'num_train_epochs': QLORA_CONFIG['training']['num_train_epochs'],
    'learning_rate': QLORA_CONFIG['training']['learning_rate'],
    'final_loss': trainer.state.log_history[-1].get('loss', 'N/A') if trainer.state.log_history else 'N/A',
    'checkpoint_validation': checkpoint_callback.validation_results if 'checkpoint_callback' in dir() else [],
}

save_adapter_only(model, tokenizer, adapter_output_dir, metadata=training_metadata)

# 注册到pipeline_state
from utils.pipeline_state import PipelineState
state = PipelineState()
state.register(
    name='adapter_checkpoint',
    path=adapter_output_dir,
    artifact_type='model',
    metadata=training_metadata,
)

print(f"\n适配器已保存到: {adapter_output_dir}")

# 查看文件
for f in os.listdir(adapter_output_dir):
    fp = os.path.join(adapter_output_dir, f)
    if os.path.isfile(fp):
        size = os.path.getsize(fp) / (1024**2)
        print(f"  {f}: {size:.2f} MB")

# 读取并显示元数据
info_path = os.path.join(adapter_output_dir, 'adapter_info.json')
if os.path.exists(info_path):
    with open(info_path, 'r') as f:
        info = json.load(f)
    print(f"\n适配器元数据:")
    for k, v in info.items():
        print(f"  {k}: {v}")

print("\n适配器保存完成!")

## 4.8 Checkpoint恢复 (可选)

如果训练中断，运行此单元格从最新checkpoint恢复。

In [ ]:
# Checkpoint恢复 — TRAIN-10
# 如果训练中断，取消注释并运行此单元格

# from utils.training_utils import resume_from_checkpoint
# 
# checkpoint_dir = QLORA_CONFIG['training']['output_dir']
# 
# # 查找最新checkpoint
# latest_checkpoint = None
# if os.path.exists(checkpoint_dir):
#     checkpoints = [d for d in os.listdir(checkpoint_dir) 
#                    if d.startswith('checkpoint-') and os.path.isdir(os.path.join(checkpoint_dir, d))]
#     if checkpoints:
#         latest = sorted(checkpoints, key=lambda x: int(x.split('-')[1]))[-1]
#         latest_checkpoint = os.path.join(checkpoint_dir, latest)
# 
# if latest_checkpoint:
#     print(f'找到最新checkpoint: {latest_checkpoint}')
#     print('恢复训练中...')
#     
#     resume_info = resume_from_checkpoint(trainer, latest_checkpoint)
#     
#     print(f'\n恢复验证:')
#     print(f'  恢复前step: {resume_info["initial_step"]}')
#     print(f'  恢复后step: {resume_info["resumed_step"]}')
#     print(f'  恢复前lr: {resume_info["initial_lr"]:.2e}')
#     print(f'  恢复后lr: {resume_info["resumed_lr"]:.2e}')
# else:
#     print('未找到checkpoint')

## 4.9 清理显存

释放GPU内存。

In [ ]:
# 清理显存
del model
del tokenizer
del trainer
if 'checkpoint_callback' in dir(): del checkpoint_callback
torch.cuda.empty_cache()

print("显存已清理")
print(f"当前显存占用: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

---

## 下一步

微调完成！请打开: **05_model_evaluation.ipynb** 评估微调效果